In [28]:
import json
import sys
import pandas as pd
import pickle
from surprise import Reader
from surprise.dataset import Dataset
from surprise import KNNBaseline
import random

 构建userid itemid rating timestamp格式的歌曲信息集

In [29]:
# 将数据转换为userid itemid rating timestamp行格式
# 观察s拆分后是否大于2项
def is_null(s):
    return len(s.split(','))>2

# 歌曲信息
def parse_song_info(song_id, score):
    try:
        song_frame = ','.join([str(song_id),str(score),'1300000']) # 1为得分
        return song_frame
    except Exception as e:
        return ""

# 用户信息+歌曲信息匹配
def parse_user_line(df_song):
    try:
        user_id = df_song.iloc[:,0]
        song_id = df_song.iloc[:,4]
        score = df_song.iloc[:,2] # 得分
        songs_info = map(lambda x:str(user_id[x])+","+parse_song_info(song_id[x], score[x]),range(len(user_id)))
        songs_info = filter(is_null, songs_info)
        return "\n".join(songs_info)
    except Exception as e:
        print(e)
        return False
    
# 读取文件+输出文件
def parse_file(in_file, out_file):
    df_song = pd.read_csv(in_file,encoding='gbk')
    result = parse_user_line(df_song)
    if(result):
        with open(out_file,'w') as out:
            out.write(result.strip()+"\n")
#    return result

In [52]:
parse_file('ranks.csv','out_song.txt')

构建用户id：用户名，歌曲id：歌曲的映射字典

In [53]:
# 用户id：用户名   歌曲id：歌曲名
def parse_user_get_info(df_user, df_song, user_dic, song_dic):
    user_name = df_user.iloc[:,1]
    user_id = df_user.iloc[:,2]
    # 用户id ：用户名
    for i in range(len(df_user)):
        user_dic[str(user_id[i])] = user_name[i] 
    # 歌曲id：歌曲名 + 歌手
    for i in range(len(df_song)):
        try:
            song_id = str(df_song.iloc[i,4])
            song_name = df_song.iloc[i,3]
            artist = df_song.iloc[i,6]
            song_dic[song_id] = song_name+"\t"+artist
        except: # 捕捉异常格式的歌曲
            print("song format error")
            print(song+"\n")
def parse_file2(in_file1, in_file2, out_user, out_song):
    user_dic = {}
    song_dic = {}
    df_user = pd.read_csv(in_file1,encoding='gbk')
    df_song = pd.read_csv(in_file2,encoding='gbk')
    parse_user_get_info(df_user, df_song, user_dic, song_dic)
    pickle.dump(user_dic, open(out_user,"wb")) 
    pickle.dump(song_dic, open(out_song,"wb"))

In [54]:
# 映射字典
in_file1 = 'D:\python\pythonProject\Bayes+snowply+lda\comments1.csv'
in_file2 = 'ranks.csv'
out_user_dic = 'out_user_dic.pkl'
out_song_dic = 'out_song_dic.pkl'
parse_file2(in_file1, in_file2, out_user_dic, out_song_dic)

 构建全训练集（此处不使用交叉验证）

In [55]:
path = './'
id_name_dic = pickle.load(open(path+"out_user_dic.pkl","rb"))
name_id_dic = {}
for user_id in id_name_dic:
    name_id_dic[id_name_dic[user_id]] = user_id

file_path = path+'out_song.txt' 
# 文本阅读器
reader = Reader(line_format='user item rating timestamp', sep=',')

# 读取数据
music_data = Dataset.load_from_file(file_path, reader=reader)
print("构建数据集...")
trainset = music_data.build_full_trainset()


构建数据集...


### 基于用户的协同过滤

训练数据集，得到近邻用户

In [56]:
print("开始训练基于用户模型...")
#sim_options = {'user_based': False}
#algo = KNNBaseline(sim_options=sim_options)
# 拟合训练集
random.seed(123)
algo = KNNBaseline()
algo.fit(trainset)

# 当前用户
current_user = list(name_id_dic.keys())[3]
# current_user = id_name_dic.get('3718199036')
print("用户", current_user)

# 取出近邻
# 映射名字到id
user_id = name_id_dic[current_user]
print("用户id", user_id)
# 取出来对应的内部user id => to_inner_uid inner id更适合surprise计算
user_inner_id = algo.trainset.to_inner_uid(user_id)
print("内部id", user_inner_id)
# 计算近邻
user_neighbors = algo.get_neighbors(user_inner_id, k=10)

# 把歌曲id转成歌曲名字
# to_raw_uid映射回去
user_neighbors = (algo.trainset.to_raw_uid(inner_id)
                       for inner_id in user_neighbors)
user_neighbors = (id_name_dic[user_id]
                       for user_id in user_neighbors)

print("和用户 [",current_user,"] 听歌风格最接近的10个用户为：\n")
for user in user_neighbors:
    print(user, algo.trainset.to_inner_uid(name_id_dic[user]))

开始训练模型...
Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
用户 爱意逃亡
用户id 3718199036
内部id 2

和用户 [ 爱意逃亡 ] 听歌风格最接近的10个用户为：

椿苓gods 66
考不上深圳中学不改名了我 286
白敬亭_VA2F 403
想睡一天的Kathy 582
smoothset 27
跨圈不糊不改名 387
青衣衫湿 393
如果看见我请让我去学习谢谢 414
顾夹 406
纾柒_ 357


In [57]:
# 相似性矩阵
pd.core.frame.DataFrame(algo.compute_similarities())


Computing the msd similarity matrix...
Done computing similarity matrix.


,0,1,2,3,4,5,6,7,8,9,...,617,618,619,620,621,622,623,624,625,626
0,1.0,0.0,0.0,0.0,0.0,0.000308,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,1.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,1.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,1.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
622,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,1.0,0.0,0.0,0.0,0.0
623,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,1.0,0.0,0.0,0.0
624,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.001657,0.0,0.0,0.0,1.0,0.0,0.0
625,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0,0.0


预测用户喜欢的歌曲

In [58]:
song_id_name_dic = pickle.load(open("out_song_dic.pkl","rb"))
# 重建歌曲名到歌曲id的映射字典
song_name_id_dic = {}
for song_id in song_id_name_dic:
    song_name_id_dic[song_id_name_dic[song_id]] = song_id

# 选择内部编码为2的用户
user_inner_id = 2
user_rating = trainset.ur[user_inner_id]
items = map(lambda x:x[0], user_rating)
for song in items:
    print(algo.predict(algo.trainset.to_raw_uid(user_inner_id), algo.trainset.to_raw_iid(song), r_ui=1), song_id_name_dic[algo.trainset.to_raw_iid(song)])


user: 3718199036 item: 517565384  r_ui = 1.00   est = 5.00   {'actual_k': 5, 'was_impossible': False} ＷＡＩＴＩＮＧ	TRA$H
user: 3718199036 item: 406072775  r_ui = 1.00   est = 5.00   {'actual_k': 5, 'was_impossible': False} Move Up (Lost Gravity)	Mr. Polska
user: 3718199036 item: 554308724  r_ui = 1.00   est = 5.00   {'actual_k': 6, 'was_impossible': False} Ephemeral?Memories	MoreanP
user: 3718199036 item: 1449116720 r_ui = 1.00   est = 5.00   {'actual_k': 3, 'was_impossible': False} ☆═━┈	HOPERUI
user: 3718199036 item: 518904426  r_ui = 1.00   est = 5.00   {'actual_k': 6, 'was_impossible': False} Umbrella (Matte Remix)	Matte
user: 3718199036 item: 515647696  r_ui = 1.00   est = 5.00   {'actual_k': 3, 'was_impossible': False} GRRRLS	AViVA


### 基于模型MF矩阵分解的协同过滤

In [96]:
from surprise import NMF

推测用户喜欢的歌曲 

In [59]:
algo1 = NMF()
trainset1 = music_data.build_full_trainset()
algo1.fit(trainset1)

user_rating1 = trainset1.ur[user_inner_id]
items1 = map(lambda x:x[0], user_rating1)
for song in items1:
    print(algo1.predict(algo1.trainset.to_raw_uid(user_inner_id), algo1.trainset.to_raw_iid(song), r_ui=1), song_id_name_dic[algo1.trainset.to_raw_iid(song)])

user: 3718199036 item: 517565384  r_ui = 1.00   est = 5.00   {'was_impossible': False} ＷＡＩＴＩＮＧ	TRA$H
user: 3718199036 item: 406072775  r_ui = 1.00   est = 2.58   {'was_impossible': False} Move Up (Lost Gravity)	Mr. Polska
user: 3718199036 item: 554308724  r_ui = 1.00   est = 2.21   {'was_impossible': False} Ephemeral?Memories	MoreanP
user: 3718199036 item: 1449116720 r_ui = 1.00   est = 2.21   {'was_impossible': False} ☆═━┈	HOPERUI
user: 3718199036 item: 518904426  r_ui = 1.00   est = 2.12   {'was_impossible': False} Umbrella (Matte Remix)	Matte
user: 3718199036 item: 515647696  r_ui = 1.00   est = 1.71   {'was_impossible': False} GRRRLS	AViVA


### 基于SVD矩阵分解的推荐系统

In [ ]:
from surprise import SVD

推测用户喜欢的歌曲

In [87]:
algo2 = SVD()
trainset2 = music_data.build_full_trainset()
algo2.fit(trainset2)

user_rating2 = trainset2.ur[user_inner_id]
items2 = map(lambda x:x[0], user_rating2)
for song in items2:
    print(algo2.predict(algo2.trainset.to_raw_uid(user_inner_id), algo2.trainset.to_raw_iid(song), r_ui=1), song_id_name_dic[algo2.trainset.to_raw_iid(song)])

user: 3718199036 item: 517565384  r_ui = 1.00   est = 5.00   {'was_impossible': False} ＷＡＩＴＩＮＧ	TRA$H
user: 3718199036 item: 406072775  r_ui = 1.00   est = 5.00   {'was_impossible': False} Move Up (Lost Gravity)	Mr. Polska
user: 3718199036 item: 554308724  r_ui = 1.00   est = 5.00   {'was_impossible': False} Ephemeral?Memories	MoreanP
user: 3718199036 item: 1449116720 r_ui = 1.00   est = 5.00   {'was_impossible': False} ☆═━┈	HOPERUI
user: 3718199036 item: 518904426  r_ui = 1.00   est = 5.00   {'was_impossible': False} Umbrella (Matte Remix)	Matte
user: 3718199036 item: 515647696  r_ui = 1.00   est = 5.00   {'was_impossible': False} GRRRLS	AViVA


## 算法评估

In [ ]:
from surprise.model_selection import cross_validate
# 五折交叉验证结果

In [60]:
# 基于用户
cross_validate(algo, music_data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
Evaluating RMSE, MAE of algorithm KNNBaseline on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    50.2800 48.3724 48.2759 48.3234 47.3317 48.5167 0.9622  
MAE (testset)     43.0408 41.3737 41.3601 41.3932 40.3204 41.4976 0.8732  
Fit time          0.02    0.02    0.02    0.02    0.02    0.02    0.00    
Test time         0.01    0.01    0.01    0.01    0.01    0.01    0.00    


{'test_rmse': array([50.279977  , 48.37242784, 48.27588571, 48.32342849, 47.33170565]),
 'test_mae': array([43.04076087, 41.37372541, 41.36005435, 41.39319728, 40.32044876]),
 'fit_time': (0.022944211959838867,
  0.015049219131469727,
  0.016913414001464844,
  0.016955137252807617,
  0.017052650451660156),
 'test_time': (0.009000539779663086,
  0.008519172668457031,
  0.008973360061645508,
  0.010976076126098633,
  0.00957345962524414)}

In [61]:
# 基于模型MF矩阵分解
cross_validate(algo1, music_data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Evaluating RMSE, MAE of algorithm NMF on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    49.4064 48.9184 48.7095 48.8465 48.5732 48.8908 0.2836  
MAE (testset)     43.0124 41.8987 41.7631 42.0360 41.7625 42.0945 0.4700  
Fit time          0.16    0.15    0.16    0.15    0.17    0.16    0.01    
Test time         0.01    0.01    0.01    0.01    0.01    0.01    0.00    


{'test_rmse': array([49.40638133, 48.91842035, 48.70952019, 48.84654319, 48.57322161]),
 'test_mae': array([43.01242105, 41.89865511, 41.76313709, 42.03600066, 41.76245027]),
 'fit_time': (0.15612292289733887,
  0.15478777885437012,
  0.16301870346069336,
  0.14630937576293945,
  0.17270708084106445),
 'test_time': (0.00892949104309082,
  0.005984067916870117,
  0.008018255233764648,
  0.0059413909912109375,
  0.006971836090087891)}

In [79]:
# 基于SVD矩阵分解
cross_validate(algo2, music_data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    48.8397 48.2410 48.9047 47.2784 49.3409 48.5209 0.7132  
MAE (testset)     41.7446 40.9457 42.0747 40.2299 42.4912 41.4972 0.8115  
Fit time          0.09    0.05    0.04    0.04    0.04    0.05    0.02    
Test time         0.01    0.01    0.01    0.01    0.01    0.01    0.00    


{'test_rmse': array([48.83968896, 48.24098812, 48.9046738 , 47.27837664, 49.34092146]),
 'test_mae': array([41.74456522, 40.94565217, 42.07472826, 40.22993197, 42.49115646]),
 'fit_time': (0.08729338645935059,
  0.048482656478881836,
  0.03785824775695801,
  0.03947806358337402,
  0.040862321853637695),
 'test_time': (0.01201176643371582,
  0.009046792984008789,
  0.005984306335449219,
  0.005955696105957031,
  0.006985664367675781)}